[Reference](https://medium.com/@tam.tamanna18/building-next-level-rag-with-strategies-for-better-ai-answers-7aae54e51b5e)

# 1. Indexing Phase

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = "Apples are red. Bananas are yellow."
splitter = RecursiveCharacterTextSplitter(chunk_size=20, chunk_overlap=5)
chunks = splitter.split_text(text)
print(chunks)

['Apples are red.', 'red. Bananas are', 'are yellow.']


# 2. Retrieval

In [2]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Embeddings (dummy vectors)
doc_emb = np.array([[0.1, 0.9], [0.9, 0.1]])  # "Apples red", "Bananas yellow"
query_emb = np.array([[0.1, 0.9]])  # "What color apples?"

similarities = cosine_similarity(query_emb, doc_emb)
top_idx = np.argmax(similarities)
print("Retrieved:", ["Apples red", "Bananas yellow"][top_idx])

Retrieved: Apples red


# 3. Generation

## 1. Multi-Query Generation

In [3]:
def multi_query(query):
    return [query, f"Examples of {query}", f"Details on {query}"]

print(multi_query("Fruit colors"))

['Fruit colors', 'Examples of Fruit colors', 'Details on Fruit colors']


## 2. RAG-Fusion
Multi-query + rank fusion (merge results).

## 3. Decomposition
Break query into sub-queries.

## 4. Step-Back Prompting
Ask a higher-level question first.

# 5. HyDE (Hypothetical Document Embeddings)
Generate a fake answer, embed it, and retrieve similar real docs.

In [4]:
def hyde(query):
    fake_doc = f"Hypothetical: {query} explained simply."
    return fake_doc  # Embed this instead

print(hyde("Fruit colors"))

Hypothetical: Fruit colors explained simply.


# Routing & Query Construction


## 1. Logical Routing
If-then rules: e.g., If query has “math”, route to calculator tool.

## 2. Semantic Routing
Use embeddings to match query to routes.

In [5]:
routes = {"weather": [0.8, 0.2], "fruits": [0.2, 0.8]}
query_emb = [0.3, 0.7]
sims = [np.dot(query_emb, r) for r in routes.values()]
best = max(sims)
print("Route to:", "fruits" if sims[1] == best else "weather")

Route to: fruits


## 3. Query Structuring
Turn natural language into structured (e.g., SQL).

# Indexing Strategies

## 1. Multi-Representation Indexing
Store summaries and full texts separately. Retrieve summaries first, then expand.


## 2. Hierarchical Indexing (RAPTOR) Knowledge Tree
Build tree: Leaves are chunks, nodes are summaries.

## 3. Token-Level Precision (ColBERT)
Embed each token, not whole chunk. Better for fine-grained matches.

# Retrieval & Generation

## 1. Dedicated Re-ranking

In [6]:
scores = [0.9, 0.7, 0.8]  # Initial
reranked = sorted(scores, reverse=True)[:2]
print(reranked)

[0.9, 0.8]


## 2. Self-Correction using AI Agents
Agent checks answer, retrieves more if wrong.

## 3. Impact of Long Context
With models like GPT-4 (128k tokens), feed more context without chunking as much.

# Manual RAG Evaluation

## 1. The Core Metrics: What Should We Measure?
- Faithfulness: Does answer match context? (No hallucinations)
- Answer Relevance: Does it answer the query?
- Context Precision: Are retrieved docs relevant?
- Context Recall: Did we get all relevant docs?

## 2. Building Evaluators from Scratch with LangChain

In [7]:
def faithfulness(answer, context):
    # Simulate LLM judge
    return 1 if context in answer else 0

print(faithfulness("Apples are red.", "Apples are red."))

1


# Evaluation with Frameworks

## 1. Rapid Evaluation with deepeval
DeepEval: Open-source, quick metrics.

In [9]:
# Pseudocode
from deepeval import evaluate
result = evaluate(rag_system, metrics=["faithfulness"])
print(result)

## 2. Another Powerful Alternative with grouse
Grouse: Focuses on groundedness.


## 3. Evaluation with RAGAS
RAGAS: Scores faithfulness, relevance, etc.

In [10]:
# Pseudocode
from ragas import evaluate
dataset = [{"query": "Fruit colors", "answer": "Red apples", "contexts": ["Apples red"]}]
scores = evaluate(dataset)
print(scores)